### Semaphore & Worker Pool — café with N espresso machines

In [ ]:
import asyncio
import time

start_time = time.monotonic()

def log(message: str) -> None:
    elapsed = time.monotonic() - start_time
    print(f"[{elapsed:6.2f}s] {message}")

In [ ]:
async def make_item(item: str = "coffee", brew: float = 2.0, fail: bool = False) -> str:
    try:
        await asyncio.sleep(brew)
    except asyncio.CancelledError:
        log(f"{item}: thrown out")
        raise
    if fail:
        raise RuntimeError(f"out of {item}")
    return item

In [ ]:
async def serve_order(order: dict, machine: asyncio.Semaphore) -> dict:
    async with machine:  # only N orders can be on a machine at once
        log(f"[order {order['order_id']}] on the machine")
        await make_item(brew=1.0)
        log(f"[order {order['order_id']}] done")
        return {"order_id": order["order_id"], "status": "ok"}


async def barista(barista_name: str, job_queue: asyncio.Queue,
                  machine: asyncio.Semaphore, finished: list):
    while True:
        order = await job_queue.get()
        if order is None:
            job_queue.task_done()
            break
        result = await serve_order(order, machine)
        finished.append(result)
        job_queue.task_done()

**5 baristas, 2 machines, 10 orders**

In [ ]:
start_time = time.monotonic()

NUM_BARISTAS = 5
NUM_MACHINES = 2

job_queue: asyncio.Queue = asyncio.Queue()
machine = asyncio.Semaphore(NUM_MACHINES)
finished: list = []

for index in range(10):
    job_queue.put_nowait({"order_id": index})
for _ in range(NUM_BARISTAS):
    job_queue.put_nowait(None)

baristas = [
    asyncio.create_task(barista(f"barista-{index}", job_queue, machine, finished))
    for index in range(NUM_BARISTAS)
]

await asyncio.gather(*baristas)
log(f"finished: {len(finished)} orders")

**`BoundedSemaphore` catches over-release**

In [ ]:
start_time = time.monotonic()

bounded_machine = asyncio.BoundedSemaphore(2)
plain_machine = asyncio.Semaphore(2)

await plain_machine.acquire()
plain_machine.release()
plain_machine.release()  # one too many — plain Semaphore silently allows it
log("plain Semaphore: over-released without complaint")

await bounded_machine.acquire()
bounded_machine.release()
try:
    bounded_machine.release()
except ValueError as error:
    log(f"BoundedSemaphore caught: {error}")

---
- `Semaphore(N)` caps how many tasks sit inside the `async with` block.
- Queue + worker pool + poison pills = clean backpressure pipeline.
- `BoundedSemaphore` raises `ValueError` on over-release.